[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# C++20: RAII, jthread y atomics

**Tema:** 02 · **Sesiones:** 9, 10 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo una operación atómica es suficiente y qué relación de memoria necesita el algoritmo?


## Resultados de aprendizaje

- Diferenciar atomicidad, orden y exclusión mutua.
- Usar RAII y `std::jthread` para administrar vida útil.
- Elegir órdenes de memoria a partir del protocolo, no por intuición.


## Modelo conceptual

Una variable atómica evita carreras sobre esa variable, pero no vuelve atómico un invariante compuesto.

Release publica escrituras anteriores y acquire permite observarlas cuando lee el valor publicado.

`memory_order_relaxed` sirve para contadores sin relación de publicación; seq_cst ofrece el modelo global más fuerte y costoso.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/03_cpp20_atomics.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Separación de contadores

Se calcula padding para evitar compartir líneas de caché cuando cada hilo escribe su contador.


In [ ]:
def padded_stride(value_bytes, cache_line=64):
    return ((value_bytes + cache_line - 1) // cache_line) * cache_line
assert padded_stride(8) == 64
assert padded_stride(72) == 128
for size in (4, 8, 16, 64, 72): print(size, padded_stride(size))


**Interpretación.** En C++ se prefiere `std::hardware_destructive_interference_size` cuando está disponible, verificando la implementación.


## Selección razonada

Una tabla relaciona patrones mínimos con el orden que debe justificarse.


In [ ]:
protocols = {
    "contador independiente": ("relaxed", "no publica otros datos"),
    "bandera de publicación": ("release/acquire", "publica y consume estado previo"),
    "algoritmo sin prueba formal": ("seq_cst", "punto de partida conservador"),
    "invariante compuesto": ("mutex", "varias ubicaciones cambian juntas"),
}
assert protocols["invariante compuesto"][0] == "mutex"
for pattern, decision in protocols.items(): print(f"{pattern:27} -> {decision[0]:15} | {decision[1]}")


**Interpretación.** La tabla no reemplaza la prueba de happens-before del algoritmo concreto.


## Práctica reproducible

1. Reescribir un contador con relaxed y justificar por qué no publica datos.
2. Modelar una bandera release/acquire con productor y consumidor.
3. Ejecutar sanitizadores y comparar con una versión protegida por mutex.


## Errores frecuentes

- Usar `volatile` como sincronización.
- Aplicar relaxed a una publicación sin relación de memoria.
- Crear hilos sin una política clara de cancelación y join.

## Criterios de aceptación

- No hay carreras en el detector.
- Cada orden de memoria tiene justificación escrita.
- La referencia protegida por mutex produce el mismo resultado.


## Referencias y material relacionado

- [Ejemplos Pthreads relacionados](../../../pthreads/)
- [Planeación C++20](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
